# Session-key protection on the sensor → QSES link

**The problem:** in the paper's protocol the sensor sends its reading to the QSES **unencrypted**, so anyone on that link can read it.

**The fix:** the sensor and the QSES run one Kyber (ML-KEM-512) handshake per session to agree on an AES key. Every reading after that is encrypted with AES-GCM. Kyber runs once per session, not once per reading.

**Who does what in the handshake:** the **sensor** makes a fresh Kyber pair (a padlock and its key), the **QSES** locks a secret with the padlock (Encaps), and the sensor opens it (Decaps).

**This notebook:**
1. Setup
2. The handshake, step by step
3. Before and after: what an eavesdropper sees
4. The reading still reaches the server
5. Attacks on the new link

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import struct

from src.crypto.kem import KEMKeyPair
from src.entities.client import ClientNode                  # the paper's sensor (sends plaintext)
from src.entities.secure_client import SecureClientNode     # new: sensor with a session key
from src.entities.secure_qses import SecureQSESNode         # new: QSES that answers the handshake
from src.entities.server import MedicalServerNode
from src.protocol.errors import ProtocolError
from src.protocol.secure_messages import LINK_PACKET_SIZE, SensorLinkPacket

print("Imports OK, project root:", project_root)

liboqs-python faulthandler is disabled
Imports OK, project root: /home/girdh/coding/experiments/pq-iomt-framework


## 1. Setup

`make_system()` creates the three machines and gives each one the keys it needs:

| Line | What it does |
|---|---|
| `qses.register_client(1, client.public_key)` | The QSES learns sensor 1's **Dilithium public key**, so it can check that sensor's signatures |
| `server.trust_qses(qses.public_key)` | The server learns the QSES's **Dilithium public key**, so it can check the QSES's signature on each packet |
| `server.register_client(1, client.public_key, client.hmac_key, 222)` | The server learns sensor 1's public key, its **HMAC key**, and the QSES nonce to expect. The QSES never gets the HMAC key |

`handshake()` runs the whole handshake in one call (the next section does it step by step).
`expect_rejected()` runs an attack and prints whether the system rejected it.

The QSES freshness window is set to 10 minutes so you can run cells by hand without hitting `stale timestamp`.

In [2]:
def make_system():
    server = MedicalServerNode(server_id=9)
    qses = SecureQSESNode(server.public_key, max_age_ms=10 * 60 * 1000)
    client = SecureClientNode(client_id=1, server_id=9, session_id=3, qses_nonce=222,
                              qses_public_key=qses.public_key)
    qses.register_client(1, client.public_key)
    server.trust_qses(qses.public_key)
    server.register_client(1, client.public_key, client.hmac_key, 222)
    return server, qses, client


def handshake(qses, client):
    cid, n_i, pk_kem, sig_d = client.begin_session()                      # step 1: sensor -> QSES
    n_q, sid, ct, sig_q = qses.accept_session(cid, n_i, pk_kem, sig_d)    # step 2: QSES -> sensor
    client.complete_session(n_q, sid, ct, sig_q)                          # step 3: sensor finishes
    return sid


def expect_rejected(action):
    try:
        action()
        print("!! NOT rejected (this would be a bug)")
    except ProtocolError as e:
        print(f"[REJECTED] {e}")

print("helpers defined")

helpers defined


## 2. The handshake (once per session)

Three steps, two messages:

| Step | Who | What happens |
|---|---|---|
| 1 | Sensor | Makes a **fresh Kyber pair**: a padlock `pk_kem` and its key `sk_kem` |
| 1 | Sensor → QSES | "I'm sensor 1. Here is a random number `n_i` and my padlock `pk_kem`, all signed by me so you know it's me." |
| 2 | QSES | Checks the signature. Then **Encaps**: locks a secret `ss` in a box `ct` using the sensor's padlock. Derives the AES key `K` from `ss` |
| 2 | QSES → Sensor | "Here is the locked box `ct`, a session label `session_id`, and my own random number `n_q`. All signed by me, including your `n_i`, so you know it's really me and it's fresh." |
| 3 | Sensor | Checks the signature. Then **Decaps**: opens the box with `sk_kem` and gets the same `ss`. Derives the same `K`, then **deletes `sk_kem`** |

Why the random numbers: an attacker cannot replay an old recorded handshake, because `n_i` is inside the QSES's signed reply.
Why the signatures: a fake box cannot pretend to be the QSES, and an attacker cannot swap in their own padlock.
Why `sk_kem` is deleted: nothing is left that could decrypt this session's key later.

In [3]:
server, qses, client = make_system()

# Step 1: sensor makes a fresh Kyber pair and sends its padlock, signed
cid, n_i, pk_kem, sig_d = client.begin_session()
print("1. sensor -> QSES :  id =", cid, "| n_i =", n_i.hex()[:8], "... | pk_kem", len(pk_kem), "B | signature", len(sig_d), "B")

# Step 2: QSES checks the signature, runs Encaps, derives K, replies
n_q, sid, ct, sig_q = qses.accept_session(cid, n_i, pk_kem, sig_d)
print("2. QSES verified the sensor signature, ran Encaps, derived K")
print("   QSES -> sensor :  ct", len(ct), "B | n_q", n_q.hex()[:8], "... | session_id", sid, "| signature", len(sig_q), "B")

# Step 3: sensor checks the signature, runs Decaps, derives K, deletes sk_kem
client.complete_session(n_q, sid, ct, sig_q)
print("3. sensor verified the QSES signature, ran Decaps, derived K and deleted sk_kem")

same_key = client._link_cipher.key == qses.sessions[sid].key
print("\nboth sides hold the same AES key:", same_key)
assert same_key

1. sensor -> QSES :  id = 1 | n_i = 7d02110c ... | pk_kem 800 B | signature 2420 B
2. QSES verified the sensor signature, ran Encaps, derived K
   QSES -> sensor :  ct 768 B | n_q 0152bddd ... | session_id 1616263414 | signature 2420 B
3. sensor verified the QSES signature, ran Decaps, derived K and deleted sk_kem

both sides hold the same AES key: True


## 3. Before and after: what an eavesdropper sees

An eavesdropper copies the raw bytes off the sensor → QSES link. It has no keys.

### Before (the paper's version)

In [4]:
baseline_sensor = ClientNode(client_id=2, server_id=9, session_id=3, qses_nonce=222)
plain = baseline_sensor.create_signed_reading(b"Heart rate: 78").to_bytes()

print("raw bytes:", plain[:40])

reading = plain[:100].rstrip(b"\x00").decode()
client_id, seq, t_ms, _, _, session, server_id = struct.unpack(">IIQIIII", plain[100:132])
print("reading :", reading)
print("device  :", client_id, "| seq:", seq, "| time(ms):", t_ms)
print("session :", session, "| destination server:", server_id)

raw bytes: b'Heart rate: 78\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
reading : Heart rate: 78
device  : 2 | seq: 1 | time(ms): 1792560195575
session : 3 | destination server: 9


### After (session key)

In [5]:
link_pkt = client.create_encrypted_reading(b"Heart rate: 78")
captured = link_pkt.to_bytes()

print("raw bytes:", captured[:40])
print("reading visible:", b"Heart rate" in captured)

assert len(captured) == LINK_PACKET_SIZE == 2616
assert b"Heart rate" not in captured
print("\nConfirmed: the eavesdropper sees only ciphertext on the sensor -> QSES link.")

raw bytes: b'`V8\xf6\xeb\x8f\xec\x08\xeb]0\xf9\xa9\x81\x8e\xd9\xcf\xdf\xa8\xab\xedNl~\xb2\xad9\xb9\x04</q(\xafv\xc0M\xdb\xa7V'
reading visible: False

Confirmed: the eavesdropper sees only ciphertext on the sensor -> QSES link.


## 4. The reading still reaches the server

The QSES decrypts the sensor's packet, then runs the paper's Algorithm 2 unchanged (freshness, replay check, sensor signature, then Kyber + AES to the server). The server verifies and stores the reading as before.

In [6]:
edge_packet = qses.receive_from_sensor(link_pkt)
stored = server.receive_and_verify(edge_packet)

print("server stored:", stored.message.rstrip(b"\x00"))
print("client_id:", stored.client_id, "| seq:", stored.seq)

server stored: b'Heart rate: 78'
client_id: 1 | seq: 1


### What the fix costs

In [7]:
print("Packet sensor -> QSES, per reading")
print("  paper   :", len(plain), "bytes")
print("  session :", len(captured), "bytes  (+", len(captured) - len(plain), ": session id, IV, GCM tag)")

print("\nHandshake, once per session")
print("  sensor -> QSES :", 4 + len(n_i) + len(pk_kem) + len(sig_d), "bytes")
print("  QSES -> sensor :", len(n_q) + 4 + len(ct) + len(sig_q), "bytes")

Packet sensor -> QSES, per reading
  paper   : 2584 bytes
  session : 2616 bytes  (+ 32 : session id, IV, GCM tag)

Handshake, once per session
  sensor -> QSES : 3240 bytes
  QSES -> sensor : 3208 bytes


## 5. Attacks on the new link

Each attack gets a fresh system, so the cells do not affect each other.

| Attack | What the attacker does | Which check stops it |
|---|---|---|
| A | Flips one bit of the ciphertext | AES-GCM tag |
| B | Changes the session id in the header | Unknown session |
| C | Replays a captured packet | Sequence number check |
| D | Sets up a rogue box pretending to be the QSES | Sensor checks the QSES signature |
| E | Replays an old recorded QSES reply | `n_i` is inside the signed reply |
| F | Sends a forged hello pretending to be the sensor | QSES checks the sensor signature |
| G | Swaps the sensor's padlock for the attacker's own | The padlock is inside the sensor's signature |
| H | Replays a recorded hello | QSES remembers used `n_i` values |
| I | n/a (sanity check) | Sensor refuses to send without a session |

## Summary

- **Fixed:** passive eavesdropping on the sensor → QSES link. Before, the reading, device id, timestamps and destination were readable. After, the eavesdropper sees ciphertext.
- **Cost:** about 32 extra bytes per reading, plus one handshake per session.
- **Preserved:** the sensor's signature still travels to the server, so end-to-end authentication is unchanged.
- **Bonus:** the sensor makes a fresh Kyber pair every session and deletes the private half, so no long-lived Kyber key on the QSES protects this link.

**Limits to state:**
- The QSES still decrypts the reading (it must, to verify the signature). This protects the link from outsiders, not from the QSES itself.
- The sensor now does KeyGen + Decaps once per session. That is slightly more work than a single Encaps, and it needs room for the 1632-byte private key during the handshake.
- The QSES → server hop still uses the server's long-term Kyber key.
- The `session_id` and `n_q` inside `meta` are still the provisioned values, because the server is unchanged.
- The QSES keeps every `n_i` it has seen, and that set grows over time.
- Timings come from a normal computer, not a real sensor.